In [1]:
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import MessagesState, StateGraph,START,END
from langchain.chat_models import init_chat_model
from rich import print as rprint
import os

from dotenv import load_dotenv
load_dotenv(override=True)

GEMINI_API_KEY=os.getenv("GEMINI_API_KEY")
GEMINI_BASE_URL=os.getenv("GEMINI_BASE_URL")

model = init_chat_model(
  model="gemini-3.1-flash-lite",
  model_provider="google_genai",
  api_key=GEMINI_API_KEY,
  transport="rest"  # 强制使用 REST 协议
)

#1. 声明状态
class OverAllState(MessagesState):
    output:str

#2. 声明节点
def llm_mode(state:OverAllState) -> OverAllState:
    messages = state["messages"]
    res = model.invoke(messages)
    return {
        "messages":[res]
    }

def output_node(state:OverAllState) -> OverAllState:
    return {
        "output":state["messages"][-1].content
    }

#3. 构建图
builder = StateGraph(state_schema=OverAllState)
builder.add_node("llm_node",llm_mode)
builder.add_node("output_node",output_node)
builder.add_edge(START,"llm_node")
builder.add_edge("llm_node","output_node")
builder.add_edge("output_node",END)

#4. 配置检查点存储器
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

#5. 使用的时候必须填写线程ID
config = {
    "configurable":{
        "thread_id":"chapter03-01"
    }
}

#6. 执行图
graph.invoke({"messages":[HumanMessage("你好,我是老王")]},config=config)


{'messages': [HumanMessage(content='你好,我是老王', additional_kwargs={}, response_metadata={}, id='448fc838-8e6c-4923-9bbe-f85c2bff7477'),
  AIMessage(content=[{'type': 'text', 'text': '你好，老王！很高兴见到你。\n\n请问今天有什么我可以帮你的吗？无论是想聊天、问问题，还是需要处理什么事情，尽管开口！', 'extras': {'signature': 'EnEKbwERTTIPK89vhzdsQOpVtWreq2Rkzxc4+fc2XFXIoZMthrc13dctz1Z36JcqXOJfQKbbkx15E/DRH0pGNkdk5EmDcKVxQAZA5lK2doStR9esNHcc7Ekk9G/dbpsQurGz1WpEkPd6LO3GxdcvhvZOxg=='}}], additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ffb38-e597-7270-9d19-5aa7b86cba01-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 6, 'output_tokens': 36, 'total_tokens': 42, 'input_token_details': {'cache_read': 0}})],
 'output': [{'type': 'text',
   'text': '你好，老王！很高兴见到你。\n\n请问今天有什么我可以帮你的吗？无论是想聊天、问问题，还是需要处理什么事情，尽管开口！',
   'extras': {'signature': 'EnE

In [2]:
graph.invoke({"messages":[HumanMessage("你好,我是谁")]},config=config)


{'messages': [HumanMessage(content='你好,我是老王', additional_kwargs={}, response_metadata={}, id='448fc838-8e6c-4923-9bbe-f85c2bff7477'),
  AIMessage(content=[{'type': 'text', 'text': '你好，老王！很高兴见到你。\n\n请问今天有什么我可以帮你的吗？无论是想聊天、问问题，还是需要处理什么事情，尽管开口！', 'extras': {'signature': 'EnEKbwERTTIPK89vhzdsQOpVtWreq2Rkzxc4+fc2XFXIoZMthrc13dctz1Z36JcqXOJfQKbbkx15E/DRH0pGNkdk5EmDcKVxQAZA5lK2doStR9esNHcc7Ekk9G/dbpsQurGz1WpEkPd6LO3GxdcvhvZOxg=='}}], additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ffb38-e597-7270-9d19-5aa7b86cba01-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 6, 'output_tokens': 36, 'total_tokens': 42, 'input_token_details': {'cache_read': 0}}),
  HumanMessage(content='你好,我是谁', additional_kwargs={}, response_metadata={}, id='877b1fca-9206-4e99-8329-3cf51ea66f0b'),
  AIMessage(co

In [8]:
config1 = {
    "configurable":{
        "thread_id":"chapter03-01xx"
    }
}
graph.invoke({"messages":[HumanMessage("你好,我是谁")]},config=config1)


{'messages': [HumanMessage(content='你好,我是谁', additional_kwargs={}, response_metadata={}, id='96a53e76-4acf-440b-99c2-2d4a76baa29e'),
  AIMessage(content='你好！很高兴见到你！不过，由于我们刚刚开始对话，我暂时还不知道你的身份信息呢。你可以告诉我你的名字或者任何你想让我知道的称呼，这样我就能更好地与你交流啦！😊', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 8, 'total_tokens': 51, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 8}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '2e946c0a-8ff9-4020-9d49-46af2a7e34dd', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f36e1-d56e-7033-8153-98ab290048b0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 43, 'total_tokens': 51, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})]